# Prueba de estructura y clasificación: ticket 31839

Resume los campos y tipos de conversación de la extracción local. No muestra `description_text`, `body_text` ni notas, porque pueden contener datos personales. Ejecuta Jupyter desde la raíz del repositorio.

In [ ]:
import json
from pathlib import Path

data_path = Path("data/inspection/ticket_31839.json")
if not data_path.is_file():
    raise FileNotFoundError("No existe la extracción local; sigue las instrucciones del README.")

payload = json.loads(data_path.read_text(encoding="utf-8"))
ticket = payload.get("ticket", {})
conversations = [item for item in payload.get("conversations", []) if isinstance(item, dict)]

summary = {
    "ticket_id": ticket.get("id"),
    "has_resolution_notes": bool((ticket.get("resolution_notes") or "").strip()),
    "conversation_count": len(conversations),
    "private_notes": sum(
        item.get("source") == 2 and item.get("private") is True
        for item in conversations
    ),
    "outgoing_public_emails": sum(
        item.get("source") == 0
        and item.get("incoming") is False
        and item.get("private") is False
        for item in conversations
    ),
}

assert ticket.get("id") == 31839
assert summary["private_notes"] >= 1, "Debe existir una nota privada clasificada como source=2."
assert summary["outgoing_public_emails"] >= 1, "Debe existir una respuesta pública saliente."
summary